# 🤖 ربات اسکالپ EUR/USD - استراتژی هیبرید

این نوت‌بوک شامل:
1. **استراتژی سفارشی شما** (قوانین EMA + RSI + Kronos AI)
2. **بک‌تست کامل** با معیارهای عملکرد
3. **اجرای زنده** با دریافت قیمت لحظه‌ای

---

In [ ]:
# =====================================================
# مرحله ۱: نصب کتابخانه‌ها (فقط بار اول)
# =====================================================

import subprocess
import sys

packages = [
    'yfinance', 'pandas', 'numpy', 'plotly', 'requests',
    'matplotlib', 'ipython'
]

for pkg in packages:
    try:
        __import__(pkg)
    except ImportError:
        print(f'📦 نصب {pkg}...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

print('✅ تمام کتابخانه‌ها نصب شدند.')

In [ ]:
# =====================================================
# مرحله ۲: وارد کردن ماژول‌ها
# =====================================================

import os
import sys

# اضافه کردن مسیر src
sys.path.insert(0, os.path.abspath('.'))

from src.strategy import HybridStrategy, SignalType, OrderType
from src.backtest import BacktestEngine, BacktestResult
from src.data_fetcher import DataFetcher
from src.visualizer import TradingVisualizer

import pandas as pd
import numpy as np
from datetime import datetime

print('✅ ماژول‌ها بارگذاری شدند.')

## ⚙️ مرحله ۳: تنظیمات استراتژی

اینجا می‌توانید پارامترهای استراتژی خود را تغییر دهید:

In [ ]:
# =====================================================
# تنظیمات استراتژی سفارشی
# =====================================================

STRATEGY_CONFIG = {
    # --- روند ---
    'trend_ema_fast': 20,         # دوره EMA سریع
    'trend_ema_slow': 50,         # دوره EMA کند
    
    # --- RSI ---
    'rsi_period': 14,             # دوره RSI
    'rsi_oversold': 30,           # سطح اشباع فروش
    'rsi_overbought': 70,         # سطح اشباع خرید
    
    # --- فیلترها ---
    'use_ema_cross': True,        # استفاده از تقاطع EMA
    'use_rsi_filter': True,       # استفاده از فیلتر RSI
    'use_kronos_prediction': False, # استفاده از Kronos AI (نیاز به نصب مدل)
    
    # --- مدیریت ریسک ---
    'risk_reward_1': 1.0,         # نسبت سود به ریسک تارگت ۱
    'risk_reward_2': 1.5,         # نسبت سود به ریسک تارگت ۲
    'risk_reward_3': 2.5,         # نسبت سود به ریسک تارگت ۳
    'atr_sl_multiplier': 1.5,     # ضریب ATR برای حد ضرر
    'max_sl_pips': 30,            # حداکثر حد ضرر (پیپ)
    'min_sl_pips': 8,             # حداقل حد ضرر (پیپ)
    
    # --- اطمینان ---
    'min_confidence': 65,         # حداقل درجه اطمینان برای ورود
    'high_confidence': 75,        # اطمینان بالا
    
    # --- ساعات معاملاتی ---
    'trading_hours_start': 8,     # شروع (UTC)
    'trading_hours_end': 20,      # پایان (UTC)
    
    # --- فیلتر اخبار ---
    'filter_high_impact_news': True,
}

# ایجاد استراتژی
strategy = HybridStrategy(config=STRATEGY_CONFIG)
print('✅ استراتژی با تنظیمات سفارشی ایجاد شد.')

## 📊 مرحله ۴: دریافت داده‌های تاریخی

In [ ]:
# =====================================================
# دریافت داده‌ها
# =====================================================

fetcher = DataFetcher()

# تنظیمات داده
SYMBOL = "EURUSD=X"
PERIOD = "1mo"      # بازه زمانی: 1d, 5d, 1mo, 3mo, 6mo, 1y
INTERVAL = "5m"     # تایم‌فریم: 1m, 5m, 15m, 30m, 1h

df = fetcher.get_historical_data(symbol=SYMBOL, period=PERIOD, interval=INTERVAL)

print(f"\n📊 خلاصه داده‌ها:")
print(f"   تعداد کندل: {len(df)}")
print(f"   از: {df['timestamps'].iloc[0]}")
print(f"   تا: {df['timestamps'].iloc[-1]}")
print(f"   قیمت فعلی: {df['close'].iloc[-1]:.5f}")

df.tail()

## 🧪 مرحله ۵: بک‌تست

In [ ]:
# =====================================================
# اجرای بک‌تست
# =====================================================

engine = BacktestEngine(
    strategy=strategy,
    initial_balance=10000,      # سرمایه اولیه ($
    risk_per_trade=0.02,        # ریسک هر معامله (2%)
    spread_pips=1.2             # اسپرد (پیپ)
)

print("⏳ در حال اجرای بک‌تست...")
result = engine.run(df, lookback=120, signal_every_n_bars=1)

print(result.summary())

In [ ]:
# =====================================================
# نمودارهای بک‌تست
# =====================================================

viz = TradingVisualizer()
viz.plot_backtest_results(result, title="نتایج بک‌تست استراتژی هیبرید - EUR/USD")

In [ ]:
# =====================================================
# تحلیل تفصیلی معاملات
# =====================================================

viz.plot_trade_analysis(result.trades, df)

# نمایش ۱۰ معامله آخر
print("\n📋 ۱۰ معامله آخر:")
for t in result.trades[-10:]:
    emoji = "🟢" if t.result == 'win' else ('🔴' if t.result == 'loss' else '⚪')
    print(f"   {emoji} {t.direction.value:4} | Entry: {t.entry_price:.5f} | Exit: {t.exit_price:.5f} | P/L: {t.pnl_pips:+.1f} pips")

## 📈 مرحله ۶: سیگنال زنده

In [ ]:
# =====================================================
# دریافت قیمت زنده و تولید سیگنال
# =====================================================

# دریافت قیمت زنده
live_price, bid, ask = fetcher.get_live_price()

if live_price:
    print(f"💹 قیمت زنده EUR/USD: {live_price:.5f}")
    print(f"   Bid: {bid:.5f} | Ask: {ask:.5f}")
    print(f"   Spread: {(ask-bid)*10000:.1f} pips")
    
    # بررسی اخبار
    has_news, news_name = fetcher.check_high_impact_news()
    if has_news:
        print(f"\n⚠️ خبر مهم نزدیک: {news_name}")
    
    # به‌روزرسانی آخرین کندل با قیمت زنده
    df_live = df.copy()
    df_live.loc[df_live.index[-1], 'close'] = live_price
    
    # تولید سیگنال
    signal = strategy.generate_signal(df_live, has_high_news=has_news)
    
    # تعیین رنگ سیگنال
    if signal.signal_type == SignalType.BUY:
        signal_color = '#06d6a0'
        signal_text = signal.reason
    elif signal.signal_type == SignalType.SELL:
        signal_color = '#ef476f'
        signal_text = signal.reason
    else:
        signal_color = '#ffd166'
        signal_text = signal.reason
    
    # تحلیل روند ۱ ساعته
    df_1h = fetcher.get_historical_data(symbol=SYMBOL, period="1mo", interval="1h")
    trend_1h = strategy.analyze_trend(df_1h)
    
    # نمایش داشبورد
    dash_data = {
        'live_price': live_price,
        'entry_price': signal.entry_price,
        'sl_price': signal.stop_loss,
        'tp1_price': signal.take_profit_1,
        'tp2_price': signal.take_profit_2,
        'tp3_price': signal.take_profit_3,
        'sl_pips': signal.sl_pips,
        'tp1_pips': signal.tp1_pips,
        'tp2_pips': signal.tp2_pips,
        'tp3_pips': signal.tp3_pips,
        'confidence': signal.confidence,
        'signal': signal_text,
        'signal_color': signal_color,
        'order_type': signal.order_type.value,
        'trend_1h_text': trend_1h['trend_text'],
        'reason': signal.reason,
    }
    
    viz.render_dashboard(dash_data)
else:
    print("⚠️ دریافت قیمت زنده ممکن نیست. استفاده از آخرین قیمت تاریخی.")
    live_price = df['close'].iloc[-1]

## 📉 مرحله ۷: نمودار قیمت با سیگنال‌ها

In [ ]:
# =====================================================
# رسم نمودار کندل‌ستیک
# =====================================================

viz.plot_candlestick_with_signals(
    df.tail(100),
    title=f"EUR/USD {INTERVAL} - ۱۰۰ کندل اخیر"
)

## 🔧 مرحله ۸: بهینه‌سازی پارامترها (اختیاری)

می‌توانید پارامترهای مختلف را تست کنید و بهترین ترکیب را پیدا کنید:

In [ ]:
# =====================================================
# تست پارامترهای مختلف
# =====================================================

param_tests = [
    {'trend_ema_fast': 10, 'trend_ema_slow': 30, 'min_confidence': 60},
    {'trend_ema_fast': 20, 'trend_ema_slow': 50, 'min_confidence': 65},
    {'trend_ema_fast': 30, 'trend_ema_slow': 100, 'min_confidence': 70},
]

results = []

for i, params in enumerate(param_tests):
    print(f"\n🧪 تست {i+1}: {params}")
    
    config = STRATEGY_CONFIG.copy()
    config.update(params)
    
    strat = HybridStrategy(config=config)
    eng = BacktestEngine(strategy=strat)
    res = eng.run(df, lookback=120)
    
    results.append({
        'params': params,
        'total_trades': res.total_trades,
        'win_rate': res.win_rate,
        'total_pnl': res.total_pnl_pips,
        'profit_factor': res.profit_factor,
        'max_dd': res.max_drawdown_pips,
        'sharpe': res.sharpe_ratio,
    })
    
    print(f"   Trades: {res.total_trades} | Win%: {res.win_rate:.1f} | P/L: {res.total_pnl_pips:.1f} | PF: {res.profit_factor:.2f}")

# نمایش خلاصه
print("\n" + "="*60)
print("📊 خلاصه تست پارامترها:")
print("="*60)
for r in results:
    print(f"\n   {r['params']}")
    print(f"   → Trades: {r['total_trades']} | Win%: {r['win_rate']:.1f}% | P/L: {r['total_pnl']:.1f} pips | PF: {r['profit_factor']:.2f} | MaxDD: {r['max_dd']:.1f}")

---

## 📝 راهنمای استفاده

### نحوه اجرا:
1. سلول‌ها را به ترتیب اجرا کنید (Shift+Enter)
2. در مرحله ۳ پارامترهای استراتژی را تنظیم کنید
3. در مرحله ۵ بک‌تست را اجرا کنید
4. در مرحله ۶ سیگنال زنده دریافت کنید

### تغییر استراتژی:
- فایل `src/strategy.py` را باز کنید
- متدهای `generate_signal` یا `_calculate_buy_levels`/`_calculate_sell_levels` را ویرایش کنید

### افزودن اندیکاتور جدید:
- در کلاس `TechnicalIndicators` متد جدید اضافه کنید
- در `generate_signal` از آن استفاده کنید